In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score, average_precision_score, confusion_matrix
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline

In [ ]:
# === 1) Load ===
def load_data(path: str):
  df = pd.read_csv(path)
  return df

In [ ]:
# === 2) Quick EDA (call this once) ===
def quick_eda(df):
  print("Shape: ", df.shape)
  print("Class Distributions:\n", df["Class"].value_counts(normalize=True))
  print("Missing Values:\n", df.isnull().sum())

In [ ]:
# === 3) Preprocess (non-ML) ===
def preprocess(df):
  df = df.copy()

  if 'Time' in df.columns:
    df = df.drop(columns=["Time"])

  # Ensure target is integer
  df["Class"] = df["Class"].astype(int)
  return df

In [ ]:
# === 4) Split ===
def split_data(df, target: str="Class"):
  X = df.drop(columns=[target])
  y = df[target]

  X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.2,random_state=42,stratify=y)
  return  X_train, X_test, y_train, y_test


In [ ]:
# === 5) Build pipeline ===
from sklearn.pipeline import Pipeline
from xgboost import XGBClassifier
def build_pipeline(classifier='rf'): # Numeric Pipeline
  # numeric cols (all except target) — dataset has only numeric features
  # We'll scale Amount; the V1..V28 are already PCA-like in Kaggle dataset but scaling is safe.
  numeric_cols = None  # we'll apply scaler to all features for safety inside ColumnTransformer

  num_tranformer = Pipeline(
      steps=[
          ("imputer", SimpleImputer(strategy="median")),
          ("scaler", StandardScaler())
      ]
  )

  def make_pipeline_for_columns(cols):
    preprocessor = ColumnTransformer(
        transformers=[
            ('num', num_tranformer, cols)
        ]
    )

    if classifier == 'rf':
      clf = RandomForestClassifier(n_estimators=200,n_jobs=-1,random_state=42)
    elif classifier == 'xgb':
      clf = XGBClassifier(n_estimators=200,use_label_encoder=False,eval_metric="logloss",random_state=42,n_jobs=-1)
    else:
      raise ValueError("Classifier must be 'rf' or 'xgb'")

    # Full pipeline
    pipe = ImbPipeline(
        steps=[
            ("preprocessor", preprocessor),
            ("smote", SMOTE(random_state=42)),
            ("classifier", clf)
        ]
    )

    return pipe

  return make_pipeline_for_columns

In [ ]:
# === 6) Train & Evaluate ===
def train_and_evaluate(pipe_maker, X_train, X_test, y_train, y_test):
  cols = X_train.columns.tolist() # Load training columns into cols variable
  pipe = pipe_maker(cols)

  print("Fitting Pipeline....")
  pipe.fit(X_train,y_train)

  y_pred = pipe.predict(X_test)
  y_proba = pipe.predict_proba(X_test)[:,-1]

  print("Classification Report:\n",classification_report(y_test, y_pred, digits=4))
  print("Confusion Matrix:\n", confusion_matrix(y_test,y_pred))

  roc_auc = roc_auc_score(y_test, y_proba)
  ap = average_precision_score(y_test, y_proba)
  print(f"ROC-AUC:{roc_auc:.4f}")
  print(f"Avg Precision:{ap:.4f}")

  return pipe

In [ ]:
def main():

  df = load_data("/content/creditcard.csv")
  quick_eda(df)
  df = preprocess(df)
  X_train, X_test, y_train, y_test = split_data(df, 'Class')

  rf_maker = build_pipeline(classifier='rf')
  rf_pipe = train_and_evaluate(rf_maker,X_train, X_test, y_train, y_test )

  xgb_maker = build_pipeline(classifier='xgb')
  xgb_pipe = train_and_evaluate(xgb_maker,X_train, X_test, y_train, y_test)

if __name__ == "__main__":
  main()